# Disneyland RAG System Demo

This notebook demonstrates the RAG system for answering questions about Disneyland visitor reviews.

## Setup

Load environment, instantiate embeddings and LLM.

In [1]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from rag.config import DATA_PATH, EMBEDDING_MODEL_NAME, LLM_MODEL_NAME, LLM_TEMPERATURE, LLM_MAX_TOKENS, LLM_TIMEOUT, EVALUATION_ENABLED
from rag.embeddings import SentenceTransformerEmbeddings
from rag.ingest import load_reviews
from rag.vectorstore import get_or_build_collection
from rag.chain import ask, AnswerWithEvaluation
from langchain_litellm import ChatLiteLLM

print(f"Data path: {DATA_PATH}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"LLM model: {LLM_MODEL_NAME}")
print(f"Evaluation: {'✅ Enabled' if EVALUATION_ENABLED else '❌ Disabled'}")

/home/syaramionak/Projects/rag-system-poc/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
22:49:06 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
22:49:06 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Data path: /home/syaramionak/Projects/rag-system-poc/data/DisneylandReviews.csv
Embedding model: all-MiniLM-L6-v2
LLM model: litellm_proxy/openrouter/openai/gpt-4.1-mini
Evaluation: ✅ Enabled


## Load and Embed Data

This cell loads reviews from CSV and embeds them into ChromaDB.
On first run, this takes 3-5 minutes. Subsequent runs load from disk instantly.

In [2]:
# Load reviews
print("Loading reviews from CSV...")
documents = load_reviews(DATA_PATH)
print(f"Loaded {len(documents)} reviews")

# Initialize embeddings
print(f"\nInitializing embeddings with {EMBEDDING_MODEL_NAME}...")
embeddings = SentenceTransformerEmbeddings()

# Build or load ChromaDB collection
print("Building/loading ChromaDB collection...")
collection = get_or_build_collection(documents, embeddings)
print(f"Collection size: {collection.count()} documents")

Loading reviews from CSV...
  (Skipped 20 duplicate review IDs)
Loaded 42636 reviews

Initializing embeddings with all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16268.01it/s]


Building/loading ChromaDB collection...

📚 EMBEDDINGS LOADED FROM CACHE (instant)
   Location: /home/syaramionak/Projects/rag-system-poc/chroma_db
   Documents: 42,636
   Status: Ready to use

Collection size: 42636 documents


## Initialize LLM

Create a ChatLiteLLM instance pointing to the LiteLLM proxy.

In [3]:
import os

proxy_url = os.getenv("LITELLM_PROXY_URL", "https://litellm.gke-prod.linnovate.net")
api_key = os.getenv("LITELLM_MASTER_KEY")

print(f"Proxy URL: {proxy_url}")
print(f"API Key: {'***' if api_key else 'NOT SET'}")

llm = ChatLiteLLM(
    model=LLM_MODEL_NAME,
    api_base=proxy_url,
    api_key=api_key,
    temperature=LLM_TEMPERATURE,
    max_tokens=LLM_MAX_TOKENS,
    timeout=LLM_TIMEOUT,
)
print("\nLLM initialized successfully")
print(f"Temperature: {LLM_TEMPERATURE}, Max tokens: {LLM_MAX_TOKENS}")

Proxy URL: https://litellm.gke-prod.linnovate.net
API Key: ***

LLM initialized successfully
Temperature: 0.7, Max tokens: 1024


In [4]:
def format_answer(result, show_evaluation=EVALUATION_ENABLED):
    """Format answer and optionally display evaluation scores."""
    if isinstance(result, dict) and "answer" in result:
        answer_text = result["answer"]
        print("Answer:")
        print(answer_text)
        
        if show_evaluation and "evaluation" in result:
            scores = result["evaluation"]
            print("\n" + "="*60)
            print("📊 Evaluation Scores:")
            print(f"   Relevance:    {scores['relevance']:.1f}")
            print(f"   Conciseness:  {scores['conciseness']:.1f}")
            print(f"   Helpfulness:  {scores['helpfulness']:.1f}")
            print(f"   Hallucination (↓ = better): {scores['hallucination']:.1f}")
    else:
        print("Answer:")
        print(result)

question_1 = "What do visitors from Australia say about Disneyland in HongKong?"
print(f"Question: {question_1}")
print("\n" + "="*60)

result_1 = ask(question_1, collection, embeddings, llm, auto_extract_filters=True, n_results=10, evaluation=EVALUATION_ENABLED)
format_answer(result_1)

In [12]:
question_1 = "What do visitors from Australia say about Disneyland in HongKong?"
print(f"Question: {question_1}")
print("\n" + "="*60)

answer_1 = ask(question_1, collection, embeddings, llm, auto_extract_filters=True, n_results=10, evaluation=True)

format_answer(answer_1)

Question: What do visitors from Australia say about Disneyland in HongKong?

Answer:
Visitors from Australia generally have positive things to say about Hong Kong Disneyland. Many highlight the joy and happiness the park brings, the friendly staff, and the clean facilities. They appreciate that the park is close to Australia and find it suitable especially for families with young children. Some mention that although the park is smaller and has fewer attractions compared to the Disneyland parks in the USA, it still offers a magical experience.

However, there are some criticisms as well. A few visitors note long queues and occasional queue-cutting by other visitors. Food is often described as overpriced and average in quality. Some mention that rides with audio are primarily in Cantonese, which can be a downside for non-Cantonese speakers. Overall, the experience is viewed as enjoyable, with a recommendation to stay at Disneyland hotels to enhance the visit.

📊 Evaluation Scores:
   Rel

question_2 = "Is spring a good time to visit Disneyland?"
print(f"Question: {question_2}")
print("\n" + "="*60)

result_2 = ask(question_2, collection, embeddings, llm, auto_extract_filters=True, n_results=10, evaluation=EVALUATION_ENABLED)
format_answer(result_2)

In [11]:
question_2 = "Is spring a good time to visit Disneyland?"
print(f"Question: {question_2}")
print("\n" + "="*60)

answer_2 = ask(question_2, collection, embeddings, llm, auto_extract_filters=True, n_results=10, evaluation=True)

format_answer(answer_2)

Question: Is spring a good time to visit Disneyland?

Answer:
Based on the visitor reviews, spring can be a mixed experience for visiting Disneyland. Some visitors mention that spring is a good time to enjoy everything, including special events like the electric light parade and fireworks. The weather in spring, especially in March and May, is often described as great or perfect, not too hot or cold.

However, several reviews also note that spring break, which falls in spring, tends to be very crowded. The park can be packed during this time, making it less enjoyable due to long lines and large crowds. Visitors suggest using Fast Pass options and apps to manage wait times during busy periods.

In summary, spring can be a good time to visit Disneyland if you avoid the peak spring break weeks and plan your visit during less busy times within the season.

📊 Evaluation Scores:
   Relevance:    1.0
   Conciseness:  0.8
   Helpfulness:  1.0
   Hallucination (↓ = better): 0.0


question_3 = "Is Disneyland California usually crowded in June?"
print(f"Question: {question_3}")
print("\n" + "="*60)

result_3 = ask(question_3, collection, embeddings, llm, auto_extract_filters=True, n_results=30, evaluation=EVALUATION_ENABLED)
format_answer(result_3)

In [13]:
question_3 = "Is Disneyland California usually crowded in June?"
print(f"Question: {question_3}")
print("\n" + "="*60)

answer_3 = ask(question_3, collection, embeddings, llm, auto_extract_filters=True, n_results=30, evaluation=True)

format_answer(answer_3)

Question: Is Disneyland California usually crowded in June?

Answer:
Based on the visitor reviews, Disneyland California is usually quite crowded in June. Multiple reviewers mention large crowds and long lines during this month. For example, one visitor noted that the crowds are huge and sometimes overbearing, while another mentioned that the park feels too small for the number of people visiting, leading to congestion. However, some suggest arriving early when the park opens to take advantage of shorter wait times before crowds build up later in the day. Overall, June tends to be a busy time at Disneyland California.

📊 Evaluation Scores:
   Relevance:    1.0
   Conciseness:  0.8
   Helpfulness:  1.0
   Hallucination (↓ = better): 0.0


question_4 = "Is the staff in Paris friendly?"
print(f"Question: {question_4}")
print("\n" + "="*60)

result_4 = ask(question_4, collection, embeddings, llm, auto_extract_filters=True, n_results=30, evaluation=EVALUATION_ENABLED)
format_answer(result_4)

In [14]:
question_4 = "Is the staff in Paris friendly?"
print(f"Question: {question_4}")
print("\n" + "="*60)

answer_4 = ask(question_4, collection, embeddings, llm, auto_extract_filters=True, n_results=30, evaluation=True)

format_answer(answer_4)

Question: Is the staff in Paris friendly?

Answer:
The reviews about the staff friendliness at Disneyland Paris are mixed:

- Several visitors from the United Kingdom and other countries have described the staff as unfriendly, rude, or unhelpful. For example, one review mentioned not meeting a single friendly member of staff, and another called the staff miserable and bolshy.
- However, other visitors have had positive experiences, describing the staff as friendly, helpful, professional, and going above and beyond to assist guests. Some noted improvement when American management was present.
- A few reviews highlighted that some specific staff members (e.g., at Starbucks or guest services) were exceptions, either positive or negative.
- Overall, while many found the staff unfriendly, others experienced very good service, suggesting inconsistency in staff friendliness.

In conclusion, the staff friendliness at Disneyland Paris seems to vary, with some guests having positive experiences 

## Debug: Inspect Retrieved Documents

For a given question, see what documents are retrieved before they go to the LLM.

In [9]:
from rag.filter_parser import extract_filters
from rag.retriever import retrieve

debug_question = "Is Disneyland California usually crowded in June?"
print(f"Debug question: {debug_question}")

# Extract filters
filters = extract_filters(debug_question, llm)
print(f"\nExtracted filters: {filters}")

# Retrieve documents with all filters
retrieved_docs = retrieve(
    debug_question,
    collection,
    embeddings,
    n_results=30,
    branch=filters.get("branch"),
    reviewer_location=filters.get("reviewer_location"),
    season=filters.get("season"),
    min_rating=filters.get("min_rating"),
    year_month=filters.get("year_month"),
    prefer_recent=filters.get("prefer_recent"),
)
print(f"\nRetrieved {len(retrieved_docs)} documents:")
print("\n" + "-"*60 + "\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}:")
    print(f"  Metadata: {doc.metadata}")
    print(f"  Text: {doc.page_content[:200]}...")
    print()

Debug question: Is Disneyland California usually crowded in June?

Extracted filters: {'branch': 'Disneyland_California', 'reviewer_location': None, 'season': 'summer', 'min_rating': None, 'year_month': '6', 'prefer_recent': False}

Retrieved 9 documents:

------------------------------------------------------------

Document 1:
  Metadata: {'rating': 5, 'branch': 'Disneyland_California', 'review_id': '133651989', 'reviewer_location': 'United States', 'year_month': '2012-6', 'season': 'summer'}
  Text: You can never go wrong with Disneyland. If you ever do travel there in June though beware of them closing early for Grad Night. I am not a fan of Grad Night. They close the Park early in order for the...

Document 2:
  Metadata: {'year_month': '2015-6', 'rating': 3, 'reviewer_location': 'United States', 'season': 'summer', 'review_id': '279373773', 'branch': 'Disneyland_California'}
  Text: Listen, we LOVE Disneyland.. we live in Utah and have annual passes.. we drive all the way just fo